In [25]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import time
import random
import math
from functools import partial
import seaborn as sns

# GerryChain imports
from gerrychain import (
    Graph, Partition, MarkovChain, proposals, updaters, constraints, accept, Election
)
from gerrychain.proposals import recom
from gerrychain.updaters import Tally, cut_edges
from gerrychain.metrics import partisan_bias, efficiency_gap, mean_median, partisan_gini

In [26]:
# Start timing
start_time = time.time()

In [27]:
# Load the Rhode Island shapefile
ri_shapefile = "./RI/RI.shp"
ri_gdf = gpd.read_file(ri_shapefile)
print(f"Loaded shapefile with {len(ri_gdf)} precincts")

Loaded shapefile with 423 precincts


In [37]:
ri_gdf.head()

,STATEFP20,COUNTYFP20,VTDST20,GEOID20,NAME20,G20PRED,G20PRER,G20USSD,G20USSR,TOTPOP,...,G18ATGOGOR,G18ATGOWRI,G18TREDMAG,G18TRERRIL,G18TREOWRI,G18SOSDGOR,G18SOSRCOR,G18SOSOWRI,SEND,geometry
0,44,009,443208,44009443208,South Kingstown 8,1562,839,1615,762,3317,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,35,"POLYGON ((282137.751 4587999.409, 282184.655 4..."
1,44,007,440724,44007440724,Cranston 24,1322,290,1310,271,2490,...,2.922391,0.162355,10.297948,5.752007,0.000000,11.156111,4.986619,0.000000,19,"POLYGON ((300198.309 4627393.802, 300267.466 4..."
2,44,005,ZZZZZZ,44005ZZZZZZ,Voting Districts Not Defined,0,0,0,0,0,...,4.466036,0.273431,15.220980,8.096589,0.015191,16.572944,7.078819,0.000000,75,"POLYGON ((289927.971 4575956.908, 290181.441 4..."
3,44,005,441501,44005441501,JAMESTOWN 1,1547,651,1621,564,2736,...,10.933789,0.692012,26.988466,21.313968,0.000000,29.525844,19.376335,0.046134,74,"POLYGON ((301064.263 4603033.208, 301072.860 4..."
4,44,005,441502,44005441502,JAMESTOWN 2,1261,562,1344,464,2823,...,1.428381,0.067218,5.192585,3.837023,0.005601,5.674313,3.288077,0.000000,74,"POLYGON ((297695.539 4591433.718, 298103.867 4..."


In [28]:
# Create the graph from the shapefile
ri_graph = Graph.from_geodataframe(ri_gdf, ignore_errors=True)
print(f"Created graph with {len(ri_graph.nodes)} nodes and {len(ri_graph.edges)} edges")

/opt/anaconda3/envs/gerry/lib/python3.11/site-packages/gerrychain/graph/graph.py:406: UserWarning: Found islands (degree-0 nodes). Indices of islands: {91}
  warnings.warn(


Created graph with 423 nodes and 1152 edges


In [29]:
# Print any islands detected
if hasattr(ri_graph, "islands"):
    print(f"Found islands (degree-0 nodes): {ri_graph.islands}")

Found islands (degree-0 nodes): {91}


In [30]:
# Define election data
election_columns = {
    "Democratic": "G20PRED", 
    "Republican": "G20PRER"
}

# Define updaters
my_updaters = {
    "population": updaters.Tally("TOTPOP", alias="population"),
    "cut_edges": cut_edges,
    "minority_pop": updaters.Tally("HISP", alias="minority_pop"),
    "election": Election("pres_2020", election_columns),
    "minority_pct": lambda p: {
        district: p["minority_pop"][district] / p["population"][district]
        for district in p["population"]
    }
}

# Create initial partition
initial_partition = Partition(
    graph=ri_graph,
    assignment="SEND",  # State Senate Districts
    updaters=my_updaters
)


In [31]:
# Print information about the initial partition
num_districts = len(initial_partition.parts)
print(f"Initial partition has {num_districts} districts")

Initial partition has 75 districts


In [32]:
# Compute ideal population and population deviation
total_population = sum(initial_partition["population"].values())
ideal_population = total_population / num_districts
print(f"Total population: {total_population}")
print(f"Ideal district population: {ideal_population:.2f}")

Total population: 1097379
Ideal district population: 14631.72


In [33]:
# Population deviation
max_pop = max(initial_partition["population"].values())
min_pop = min(initial_partition["population"].values())
max_deviation = (max_pop - ideal_population) / ideal_population
min_deviation = (ideal_population - min_pop) / ideal_population
print(f"Maximum population deviation: {max_deviation:.2%}")
print(f"Minimum population deviation: {min_deviation:.2%}")

Maximum population deviation: 27.41%
Minimum population deviation: 16.51%


In [34]:
sum(initial_partition["pres_2020"].counts("Democratic"))

307486

In [36]:
# Analyze partisan balance
dem_votes = sum(initial_partition["pres_2020"].counts("Democratic"))
rep_votes = sum(initial_partition["pres_2020"].counts("Republican"))
total_votes = dem_votes + rep_votes
dem_seat_share = sum(1 for vote_share in initial_partition["pres_2020"].percents("Democratic") if vote_share > 0.5) / num_districts
dem_vote_share = dem_votes / total_votes

print(f"Democratic vote share: {dem_vote_share:.2%}")
print(f"Democratic seat share: {dem_seat_share:.2%}")
print(f"Efficiency Gap: {efficiency_gap(initial_partition, 'pres_2020'):.4f}")
print(f"Mean-Median Difference: {mean_median(initial_partition, 'pres_2020'):.4f}")

Democratic vote share: 60.60%
Democratic seat share: 84.00%


AttributeError: 'ElectionResults' object has no attribute 'parties'

In [ ]:
# Analyze racial composition
minority_districts = sum(1 for pct in initial_partition["minority_pct"].values() if pct > 0.5)
print(f"Number of majority-minority districts: {minority_districts}")

In [ ]:
# Set up ReCom proposal
recom_proposal = partial(
    recom,
    pop_col="TOTPOP",
    pop_target=ideal_population,
    epsilon=0.05,  # Population balance constraint (5%)
    node_repeats=2
)

In [ ]:
# Define constraints
pop_constraint = constraints.within_percent_of_ideal_population(initial_partition, 0.1)  # 10% population deviation

In [ ]:
# Configure the Markov chain
num_steps = 10000
chain = MarkovChain(
    proposal=recom_proposal,
    constraints=[pop_constraint],
    accept=accept.always_accept,
    initial_state=initial_partition,
    total_steps=num_steps
)

In [ ]:
# Set up data collection
data = {
    "cut_edges": [],
    "dem_seats": [],
    "minority_seats": [],
    "efficiency_gap": [],
    "mean_median": [],
    "partisan_gini": []
}

In [ ]:
# Run the chain
print(f"Running {num_steps} steps of Markov chain...")
for i, partition in enumerate(chain):
    if i % 1000 == 0:
        print(f"Step {i} of {num_steps}")
    
    # Save data
    data["cut_edges"].append(len(partition["cut_edges"]))
    data["dem_seats"].append(sum(1 for vote_share in partition["pres_2020"].percents("Democratic") if vote_share > 0.5))
    data["minority_seats"].append(sum(1 for pct in partition["minority_pct"].values() if pct > 0.5))
    data["efficiency_gap"].append(efficiency_gap(partition, "pres_2020"))
    data["mean_median"].append(mean_median(partition, "pres_2020"))
    data["partisan_gini"].append(partisan_gini(partition, "pres_2020"))


In [ ]:
df = pd.DataFrame(data)

# statistics
print("\nSummary Statistics:")
print(f"Mean number of cut edges: {df['cut_edges'].mean():.2f}")
print(f"Mean Democratic seats: {df['dem_seats'].mean():.2f}")
print(f"Mean majority-minority seats: {df['minority_seats'].mean():.2f}")
print(f"Mean Efficiency Gap: {df['efficiency_gap'].mean():.4f}")
print(f"Mean Mean-Median Difference: {df['mean_median'].mean():.4f}")
print(f"Mean Partisan Gini: {df['partisan_gini'].mean():.4f}")

In [ ]:
# Create plots
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

axes[0, 0].hist(df["cut_edges"], bins=20, edgecolor="black")
axes[0, 0].axvline(len(initial_partition["cut_edges"]), color="red", linestyle="--", label="Initial Plan")
axes[0, 0].set_title("Number of Cut Edges")
axes[0, 0].set_xlabel("Cut Edges")
axes[0, 0].set_ylabel("Frequency")
axes[0, 0].legend()

In [ ]:
# Democratic seats
axes[0, 1].hist(df["dem_seats"], bins=range(num_districts + 2), edgecolor="black")
initial_dem_seats = sum(1 for vote_share in initial_partition["pres_2020"].percents("Democratic") if vote_share > 0.5)
axes[0, 1].axvline(initial_dem_seats, color="red", linestyle="--", label="Initial Plan")
axes[0, 1].set_title("Democratic Seats")
axes[0, 1].set_xlabel("Number of Democratic Seats")
axes[0, 1].set_ylabel("Frequency")
axes[0, 1].legend()

In [ ]:
# Majority-minority seats
axes[0, 2].hist(df["minority_seats"], bins=range(max(df["minority_seats"].max() + 2, 2)), edgecolor="black")
initial_minority_seats = sum(1 for pct in initial_partition["minority_pct"].values() if pct > 0.5)
axes[0, 2].axvline(initial_minority_seats, color="red", linestyle="--", label="Initial Plan")
axes[0, 2].set_title("Majority-Minority Seats")
axes[0, 2].set_xlabel("Number of Majority-Minority Seats")
axes[0, 2].set_ylabel("Frequency")
axes[0, 2].legend()

In [ ]:
# Efficiency Gap
axes[1, 0].hist(df["efficiency_gap"], bins=20, edgecolor="black")
initial_eg = efficiency_gap(initial_partition, "pres_2020")
axes[1, 0].axvline(initial_eg, color="red", linestyle="--", label="Initial Plan")
axes[1, 0].set_title("Efficiency Gap")
axes[1, 0].set_xlabel("Efficiency Gap")
axes[1, 0].set_ylabel("Frequency")
axes[1, 0].legend()

In [ ]:
# Mean-Median Difference
axes[1, 1].hist(df["mean_median"], bins=20, edgecolor="black")
initial_mm = mean_median(initial_partition, "pres_2020")
axes[1, 1].axvline(initial_mm, color="red", linestyle="--", label="Initial Plan")
axes[1, 1].set_title("Mean-Median Difference")
axes[1, 1].set_xlabel("Mean-Median Difference")
axes[1, 1].set_ylabel("Frequency")
axes[1, 1].legend()

In [ ]:
# Partisan Gini
axes[1, 2].hist(df["partisan_gini"], bins=20, edgecolor="black")
initial_pg = partisan_gini(initial_partition, "pres_2020")
axes[1, 2].axvline(initial_pg, color="red", linestyle="--", label="Initial Plan")
axes[1, 2].set_title("Partisan Gini")
axes[1, 2].set_xlabel("Partisan Gini")
axes[1, 2].set_ylabel("Frequency")
axes[1, 2].legend()

In [ ]:
plt.tight_layout()
plt.savefig("ri_recom_results.png", dpi=300)
plt.show()

In [ ]:
# Plot Democratic vote share vs seat share
# Ensemble results
ensemble_vote_shares = [dem_votes / total_votes] * len(df)
ensemble_seat_shares = df["dem_seats"] / num_districts

In [ ]:
plt.figure(figsize=(10, 8))
plt.scatter(ensemble_vote_shares, ensemble_seat_shares, alpha=0.3, label="Ensemble Plans")
plt.scatter(dem_vote_share, dem_seat_share, color="red", s=100, marker="*", label="Initial Plan")

In [ ]:
# Plot seats-votes curve (theoretical proportional representation)
x = np.linspace(0, 1, 100)
y = x
plt.plot(x, y, "k--", label="Proportional Representation")

In [ ]:
plt.xlabel("Democratic Vote Share")
plt.ylabel("Democratic Seat Share")
plt.title("Seats-Votes Curve")
plt.xlim(0, 1)
plt.ylim(0, 1)
plt.grid(True, alpha=0.3)
plt.legend()
plt.savefig("ri_seats_votes.png", dpi=300)
plt.show()

In [ ]:
# Box plots of vote shares by district
all_districts_vote_shares = []
district_labels = []

for i, partition in enumerate(chain):
    if i % 100 == 0:  # Sample every 100th plan to avoid overwhelming the plot
        vote_shares = sorted(partition["pres_2020"].percents("Democratic"))
        all_districts_vote_shares.append(vote_shares)
        district_labels = [f"District {i+1}" for i in range(len(vote_shares))]


In [ ]:
# Convert to format suitable for box plot
box_data = []
for i in range(num_districts):
    district_data = [plan[i] for plan in all_districts_vote_shares]
    box_data.append(district_data)

In [ ]:
plt.figure(figsize=(12, 8))
box = plt.boxplot(box_data, labels=district_labels, patch_artist=True)

# Color the boxes based on median value
for patch, median in zip(box['boxes'], [np.median(x) for x in box_data]):
    if median > 0.5:
        patch.set_facecolor('lightblue')
    else:
        patch.set_facecolor('pink')

# Add a horizontal line at 0.5
plt.axhline(0.5, color='red', linestyle='--', alpha=0.7, label='50% Democratic Vote Share')

plt.title('Distribution of Democratic Vote Share by District (Sorted)')
plt.ylabel('Democratic Vote Share')
plt.grid(True, axis='y', alpha=0.3)
plt.legend()
plt.savefig("ri_boxplot_districts.png", dpi=300)
plt.show()

In [ ]:
# Calculate and display runtime
end_time = time.time()
print(f"Runtime: {(end_time - start_time)/60:.2f} minutes")